# S10 – Feature analysis and WoE binning
Training loans only (131,366). Checks the fields, fills blanks, looks at drift by year, bins 16 candidate features with WoE, checks the bins on validation and picks 9 for the scorecard. Decisions: D-058 to D-063 in docs/DECISIONS.md.

## 1. Setup and load the training loans

In [ ]:
import os  # reads settings from the environment

import pandas as pd  # pandas, always nicknamed "pd"
from dotenv import load_dotenv  # loads .env, same as loader.py
from sqlalchemy import URL, create_engine  # the connection pandas prefers

pd.set_option("display.max_columns", None)  # show all columns, never "..."

load_dotenv()

url = URL.create(
    "postgresql+psycopg",                   # Postgres, through psycopg
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),         # must be a number here, not text
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)                 # a reusable door into the database

In [ ]:
query = "SELECT * FROM pd_features_build WHERE sample_set = 'train'"
df = pd.read_sql(query, engine)   # run the query, keep the result as a table
df.shape                          # (rows, columns)

## 2. First look
head, info and describe: what each column holds, its type, its blanks and its range.

In [ ]:
print(df.head(10).T.to_string())

In [ ]:
df.info()

In [ ]:
print(df.describe().T.to_string())

## 3. Field checks (D-060)
Fields that carry no information, or that need care before use.

In [ ]:
for col in ["amortization_type", "prepayment_penalty_indicator",
            "interest_only_indicator", "property_type", "channel"]:
    print(df[col].value_counts(dropna=False))
    print()

In [ ]:
print(df["msa"].isna().mean())
print(df["msa"].nunique())
print(df["msa"].value_counts().head(10))

In [ ]:
print(df["postal_code"].nunique())
print(df["property_state"].nunique())

In [ ]:
print("--- check 3: LTV over 100 ---")
high_ltv = df[df["original_ltv"] > 100]
print(len(high_ltv))
print(high_ltv["harp_indicator"].value_counts())

print("--- check 4: rate spread range, and the odd 276/288 terms ---")
print(df["rate_spread"].describe())
print(df[df["loan_term"].isin([276, 288])]["harp_indicator"].value_counts())

print("--- check 5: credit scores at the very bottom ---")
print((df["credit_score"] == 300).sum())
print((df["credit_score"] < 500).sum())

## 4. Missing values (D-061)
Blanks are filled with the typical training value (median or most common), so validation, test and 2017 get the same fills.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(3)}))

In [ ]:
fill_numbers = ["credit_score", "number_of_borrowers", "number_of_units",
                "original_cltv", "original_ltv", "mi_percentage"]
print(df[fill_numbers].median())
print(df["first_time_homebuyer"].mode())
print(df["property_type"].mode())

print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))


## 5. Drift by year and HARP (D-062)
How default rates and fields change by year. HARP refinances looked safe overall but were 10 times riskier within 2012, so they left the model's scope.

In [ ]:
df["year"] = "20" + df["loan_sequence_number"].str[1:3]
print(df.groupby("year")["target"].agg(["count", "mean"]))

In [ ]:
num_cols = ["credit_score", "original_ltv", "original_cltv", "original_dti_ratio",
            "original_upb", "rate_spread", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units"]
print(df.groupby("year")[num_cols].mean().round(2).T)

In [ ]:
for col in ["loan_purpose", "channel", "occupancy_status", "property_type",
            "first_time_homebuyer", "harp_indicator"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

In [ ]:
for col in ["property_type", "first_time_homebuyer"]:
    print(pd.crosstab(df["year"], df[col], normalize="index").round(3))
    print()

y2012 = df[df["year"] == "2012"]
print(y2012.groupby("harp_indicator")[["target", "rate_spread"]].mean())

In [ ]:
print((df["original_ltv"] > 100).sum())

In [ ]:
print(df.groupby(df["original_dti_ratio"].isna())["target"].agg(["count", "mean"]))

## 6. Training table: 16 candidate features

In [ ]:
# The model's features: D-057 candidates plus rate_spread,
# minus the 5 fields of D-060 and harp_indicator (D-062)
features = ["credit_score", "original_ltv", "original_cltv", "original_dti_ratio",
            "original_upb", "rate_spread", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units", "first_time_homebuyer",
            "occupancy_status", "channel", "property_state", "property_type",
            "loan_purpose"]
train = df[["target"] + features].copy()

# D-062: channel as retail (R) vs third party (TP)
train["channel"] = train["channel"].replace({"B": "TP", "C": "TP", "T": "TP"})

# D-061: fill small gaps with values learned on train
fill_values = train[["credit_score", "original_ltv", "original_cltv",
                     "original_dti_ratio", "mi_percentage",
                     "number_of_borrowers", "number_of_units"]].median().to_dict()
fill_values["first_time_homebuyer"] = train["first_time_homebuyer"].mode()[0]
fill_values["property_type"] = train["property_type"].mode()[0]
train = train.fillna(fill_values)

print(train.shape)
print(fill_values)
print(train.isna().sum().sum())
print(train["channel"].value_counts())

## 7. WoE binning and IV on train
At least 5% of loans per bin, at most 8 bins, risk moving one way only.

In [ ]:
from optbinning import OptimalBinning

optb = OptimalBinning(name="credit_score", dtype="numerical",
                      monotonic_trend="auto_asc_desc",
                      min_bin_size=0.05, max_n_bins=8)
optb.fit(train["credit_score"], train["target"])

table = optb.binning_table.build()
print(table[["Bin", "Count", "Count (%)", "Non-event", "Event",
             "Event rate", "WoE", "IV"]].to_string())

In [ ]:
binners = {"credit_score": optb}

for col in ["original_ltv", "original_dti_ratio", "rate_spread", "original_upb"]:
    ob = OptimalBinning(name=col, dtype="numerical",
                        monotonic_trend="auto_asc_desc",
                        min_bin_size=0.05, max_n_bins=8)
    ob.fit(train[col], train["target"])
    binners[col] = ob
    t = ob.binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()

In [ ]:
for col in ["original_cltv", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units"]:
    ob = OptimalBinning(name=col, dtype="numerical",
                        monotonic_trend="auto_asc_desc",
                        min_bin_size=0.05, max_n_bins=8)
    ob.fit(train[col], train["target"])
    binners[col] = ob
    t = ob.binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()

In [ ]:
# optbinning wants text categories, so True/False become "True"/"False".
# Validation, test and 2017 get the same change later.
train["first_time_homebuyer"] = train["first_time_homebuyer"].astype(str)

for col in ["first_time_homebuyer", "occupancy_status", "channel",
            "property_type", "loan_purpose", "property_state"]:
    ob = OptimalBinning(name=col, dtype="categorical",
                        min_bin_size=0.05, max_n_bins=8)
    ob.fit(train[col], train["target"])
    binners[col] = ob
    t = ob.binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()

In [ ]:
for col in ["loan_purpose", "property_state"]:
    t = binners[col].binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()

In [ ]:
for col in ["original_cltv", "loan_term", "mi_percentage",
            "number_of_borrowers", "number_of_units"]:
    t = binners[col].binning_table.build()
    print(col, "IV =", round(t.loc["Totals", "IV"], 3))
    print(t[["Bin", "Count (%)", "Event rate", "WoE"]].iloc[:-3].to_string())
    print()

## 8. Validation check and rebinning
Bins fitted on train, checked on validation. Five features whose bins swapped order were rebinned (at least 10% of loans per bin, neighbours at least 0.3 points apart).

In [ ]:
import numpy as np

# Validation rows, prepared exactly like train, with train's fill values
query = "SELECT * FROM pd_features_build WHERE sample_set = 'validation'"
val = pd.read_sql(query, engine)
val = val[["target"] + features].copy()
val["channel"] = val["channel"].replace({"B": "TP", "C": "TP", "T": "TP"})
val = val.fillna(fill_values)
val["first_time_homebuyer"] = val["first_time_homebuyer"].astype(str)
print(val.shape)

# IV and default rate per bin, on any set, using bins already learned
def iv_on(ob, x, y):
    bins = ob.transform(x, metric="indices")
    t = pd.DataFrame({"bin": bins, "y": y.to_numpy()})
    g = t.groupby("bin")["y"].agg(["count", "sum"])
    bad = g["sum"]
    good = g["count"] - g["sum"]
    pg, pb = good / good.sum(), bad / bad.sum()
    return ((pg - pb) * np.log(pg / pb)).sum(), bad / g["count"]

rows = []
for col in features:
    ob = binners[col]
    iv_tr, rate_tr = iv_on(ob, train[col], train["target"])
    iv_va, rate_va = iv_on(ob, val[col], val["target"])
    rate_va = rate_va.reindex(rate_tr.index)
    same = bool((np.sign(np.diff(rate_tr.to_numpy()))
                 == np.sign(np.diff(rate_va.to_numpy()))).all())
    rows.append({"feature": col, "iv_train": round(iv_tr, 3),
                 "iv_val": round(iv_va, 3), "same_order": same})

result = pd.DataFrame(rows).sort_values("iv_train", ascending=False)
print(result.to_string(index=False))

In [ ]:
for col in ["rate_spread", "original_cltv", "original_dti_ratio",
            "property_state", "original_upb"]:
    _, rate_tr = iv_on(binners[col], train[col], train["target"])
    _, rate_va = iv_on(binners[col], val[col], val["target"])
    print(col)
    side = pd.DataFrame({"train": rate_tr.round(4), "val": rate_va.round(4)})
    print(side.to_string())
    print()

In [ ]:
for col in ["rate_spread", "original_cltv", "original_dti_ratio",
            "property_state", "original_upb"]:
    if col == "property_state":
        ob = OptimalBinning(name=col, dtype="categorical", max_n_bins=8,
                            min_bin_size=0.10, min_event_rate_diff=0.003)
    else:
        ob = OptimalBinning(name=col, dtype="numerical", max_n_bins=8,
                            monotonic_trend="auto_asc_desc",
                            min_bin_size=0.10, min_event_rate_diff=0.003)
    ob.fit(train[col], train["target"])
    binners[col] = ob
    iv_tr, rate_tr = iv_on(ob, train[col], train["target"])
    iv_va, rate_va = iv_on(ob, val[col], val["target"])
    print(col, "| IV train", round(iv_tr, 3), "| IV val", round(iv_va, 3))
    side = pd.DataFrame({"train": rate_tr.round(4), "val": rate_va.round(4)})
    print(side.to_string())
    print()

## 9. Twins and the final 9 (D-063)

In [ ]:
twins = ["original_ltv", "original_cltv", "mi_percentage"]
woe = pd.DataFrame({c: binners[c].transform(train[c], metric="woe") for c in twins})
print(woe.corr().round(2))
print()

state_bin = binners["property_state"].transform(
    train["property_state"], metric="indices"
)
by_year = pd.DataFrame({"bin": state_bin, "year": df["year"].to_numpy(),
                        "target": train["target"].to_numpy()})
print(by_year.pivot_table(index="bin", columns="year",
                          values="target", aggfunc="mean").round(4))

In [ ]:
# Save the 9 final binning tables: summary counts only, no loan rows
from pathlib import Path

final_9 = [
    "credit_score",
    "original_ltv",
    "rate_spread",
    "original_dti_ratio",
    "mi_percentage",
    "number_of_borrowers",
    "loan_term",
    "property_state",
    "channel",
]
tables = []
for col in final_9:
    t = binners[col].binning_table.build()
    t.insert(0, "feature", col)
    tables.append(t)
    print(f"{col:22} bins={len(t) - 3:2}  IV={t.loc['Totals', 'IV']:.2f}")

out = Path("../docs/s10_binning_tables.csv")
pd.concat(tables).to_csv(out)
print("saved to", out.resolve())